# Notebook 16 — Cross-Validation: Zero-Shot Transfer Across All DO Gauges

Tests whether zero-shot transfer works across all training source gauges,
not just gauge 2473. Leave-one-out design: train on each gauge, transfer to all others.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

REPO_ROOT = Path('/storage/homefs/tn20y076/AareML')
sys.path.insert(0, str(REPO_ROOT))

from src.config import (FEATURES, TARGETS, LOOKBACK, HORIZON,
                         TRAIN_END, VAL_END, SEED, RESULTS_DIR, FIGURES_DIR)
from src.data import load_gauge, preprocess, train_val_test_split, make_windows
from src.model import Seq2SeqLSTM, RiverDataset, train_model, predict, get_y_true
from src.metrics import mean_rmse, nse

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Hyperparameters (same as nb03)
HIDDEN, N_LAYERS, DROPOUT = 128, 2, 0.3
LR, EPOCHS, PATIENCE = 1e-3, 50, 8

# All 16 DO gauges sorted by DO coverage
DO_GAUGES = [2473, 2009, 2613, 2143, 2016, 2174, 2415,
             2044, 2410, 2085, 2462, 2018, 2243, 2068, 2135, 2130]
print(f'DO gauges: {len(DO_GAUGES)}')

## 1. Leave-One-Out Cross-Validation

In [ ]:
# Pre-load all gauge data to avoid repeated IO
print('Loading all gauge data...')
gauge_data = {}
for gid in tqdm(DO_GAUGES):
    try:
        raw = load_gauge(gid)
        data = preprocess(raw)
        train_df, val_df, test_df = train_val_test_split(data)
        train_means = (pd.concat([train_df[FEATURES].mean(), train_df[TARGETS].mean()])
                       .groupby(level=0).first())
        X_te, y_te, _ = make_windows(test_df, train_means,
                                      features=FEATURES, targets=TARGETS)
        gauge_data[gid] = {
            'train': train_df, 'val': val_df, 'test': test_df,
            'means': train_means, 'X_te': X_te, 'y_te': y_te,
        }
        print(f'  {gid}: {len(X_te)} test windows')
    except Exception as e:
        print(f'  {gid}: FAILED ({e})')

print(f'\nLoaded: {len(gauge_data)}/{len(DO_GAUGES)} gauges')

In [ ]:
DO_IDX = list(TARGETS).index('O2C_sensor')
results = []

for src_gid in tqdm(DO_GAUGES, desc='Source gauges'):
    if src_gid not in gauge_data:
        continue
    
    # Train model on source gauge
    gd = gauge_data[src_gid]
    ds_tr = RiverDataset(gd['train'], gd['means'], FEATURES, TARGETS)
    ds_va = RiverDataset(gd['val'],   gd['means'], FEATURES, TARGETS)
    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=256, shuffle=True)
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=256, shuffle=False)
    
    model = Seq2SeqLSTM(
        n_feat=len(FEATURES), n_tgt=len(TARGETS),
        hidden=HIDDEN, n_layers=N_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)
    model, _ = train_model(model, dl_tr, dl_va, device=DEVICE,
                            epochs=EPOCHS, lr=LR, patience=PATIENCE, verbose=False)
    
    # Zero-shot transfer to all other gauges
    for tgt_gid in DO_GAUGES:
        if tgt_gid == src_gid or tgt_gid not in gauge_data:
            continue
        tgt = gauge_data[tgt_gid]
        if len(tgt['X_te']) < 10:
            continue
        try:
            ds_te = RiverDataset(tgt['test'], tgt['means'], FEATURES, TARGETS)
            dl_te = torch.utils.data.DataLoader(ds_te, batch_size=256, shuffle=False)
            y_pred = predict(model, dl_te, DEVICE)
            y_true = get_y_true(ds_te)
            
            rmse_val = mean_rmse(y_true, y_pred)[TARGETS[DO_IDX]]
            nse_val  = nse(y_true, y_pred)[TARGETS[DO_IDX]]
            
            results.append({
                'source_gauge': src_gid,
                'target_gauge': tgt_gid,
                'rmse_do': round(rmse_val, 4),
                'nse_do':  round(nse_val, 3),
            })
        except Exception as e:
            print(f'  {src_gid}\u2192{tgt_gid}: {e}')

df_cv = pd.DataFrame(results)
print(f'\nTotal pairs: {len(df_cv)}')
print(f'Mean RMSE:   {df_cv["rmse_do"].mean():.4f} mg/L')
print(f'Mean NSE:    {df_cv["nse_do"].mean():.3f}')

# Comparison: fixed source 2473 vs all other sources
src_2473 = df_cv[df_cv['source_gauge'] == 2473]
src_other = df_cv[df_cv['source_gauge'] != 2473]
print(f'\nSource=2473:  mean RMSE={src_2473["rmse_do"].mean():.4f}  NSE={src_2473["nse_do"].mean():.3f}')
print(f'Other sources: mean RMSE={src_other["rmse_do"].mean():.4f}  NSE={src_other["nse_do"].mean():.3f}')

df_cv.to_csv(RESULTS_DIR / 'cv_transfer_results.csv', index=False)
print(f'Saved: {RESULTS_DIR}/cv_transfer_results.csv')

## 2. Results Heatmap

In [ ]:
# Pivot to matrix
pivot = df_cv.pivot(index='source_gauge', columns='target_gauge', values='rmse_do')

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r',
            vmin=0.2, vmax=1.5, ax=ax,
            cbar_kws={'label': 'RMSE (mg/L DO)'})
ax.set_title('Zero-Shot Transfer RMSE: Source \u2192 Target Gauge\n(lower = better transfer)',
             fontsize=13)
ax.set_xlabel('Target Gauge'); ax.set_ylabel('Source Gauge')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb16_cv_heatmap.png', dpi=150)
plt.close()
print('Saved: nb16_cv_heatmap.png')

## 3. Per-Source Summary

In [ ]:
src_summary = df_cv.groupby('source_gauge')['rmse_do'].mean().sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#01696F' if g == 2473 else '#D4D1CA' for g in src_summary.index]
ax.bar(src_summary.index.astype(str), src_summary.values, color=colors)
ax.axhline(src_summary.mean(), color='#d62728', linestyle='--', linewidth=1.5,
           label=f'Mean across all sources: {src_summary.mean():.3f} mg/L')
ax.set_xlabel('Source Gauge'); ax.set_ylabel('Mean Transfer RMSE (mg/L)')
ax.set_title('Mean Zero-Shot Transfer RMSE by Source Gauge\n(teal = gauge 2473, our chosen training gauge)')
ax.legend(); plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb16_source_comparison.png', dpi=150)
plt.close()
print('Saved: nb16_source_comparison.png')

## 4. Summary

In [ ]:
print('=' * 60)
print('CROSS-VALIDATION SUMMARY')
print('=' * 60)
print(f'Total transfer pairs:     {len(df_cv)}')
print(f'Overall mean RMSE:        {df_cv["rmse_do"].mean():.4f} mg/L')
print(f'Overall mean NSE:         {df_cv["nse_do"].mean():.3f}')
print(f'')
print(f'Source=2473 mean RMSE:    {src_2473["rmse_do"].mean():.4f} mg/L')
print(f'All-source mean RMSE:     {df_cv["rmse_do"].mean():.4f} mg/L')
print(f'')
print(f'Best source gauge:        {src_summary.index[0]} (RMSE={src_summary.iloc[0]:.4f})')
print(f'Worst source gauge:       {src_summary.index[-1]} (RMSE={src_summary.iloc[-1]:.4f})')
print(f'')
print(f'Interpretation: {"gauge 2473 transfers better than average" if src_2473["rmse_do"].mean() < df_cv["rmse_do"].mean() else "gauge 2473 is representative of average transfer performance"}')